#<mark> 5. 한국어 또는 영어 챗봇을 만드세요.


~~1. 사용자가 문장을 넣으면 다음 단어를 생성하는 모델을 만드세요.~~

~~2. 그 모델을 재귀 반복적(auto regressive)으로 사용하여 완전한 문장을 생성하세요.~~

3. FastAPI로 감싸서 웹에서 사용할 수 있도록 하세요.

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 🤖 : "나는 학교에 간다"를 글자 단위로 쪼갤 때, 공백도 하나의 토큰으로 포함시킨다고 하면 — 토큰이 총 몇 개가 될까?
# 🐥 : 공백을 하나라고 쳤을 때, 중복으로 나온 것도 결국 같은 공백이니까 나,는,(공백),학,교,에,간,다 로 8개 맞아?
# 🤖 : 절반은 맞고, 절반은 틀려! 하지만, 아주 잘했어
# 1) vocab 크기 |   중복 제거  | 8개
# 2) 시퀀스 길이  | 위치마다 센다 | 9개
# 수빈의 "8"은 vocab 크기로는 완벽한 정답이야.
# 다만 "중복 제거"라는 한정어는 vocab을 셀 때만 붙어. 시퀀스는 위치마다 따로 세니까 공백 2개를 둘로 친다.
# nn.Embedding(num_embeddings = vocab크기, embedding_dim)      ← 사전 테이블 행 수 (고유)
# 입력 텐서 shape = (batch, 시퀀스길이)                            ← 시간축 길이 (위치)
#
# 🤖 : 이 8개 vocab으로 임베딩 층을 만들면 nn.Embedding(num_embeddings=?, embedding_dim=...) 의 ? 자리엔 8과 9 중 뭘 넣어야 할까?
# 🐥 : 9를 넣어야 하나..?
# 🤖 : 아냐. 8이야. num_embeddings는 "몇 종류냐"(고유 = vocab = 8)에 달렸지, "몇 번 등장하냐"(위치 = 시퀀스 = 9)가 아니야.
# 8은 사전에, 9는 입력 길이에. 이 그림을 머리에 박고 시작하자.
# ---------------------------------------------------------------------------------------------

 # <font color='purple'>시작 전 예제

In [ ]:
text = "나는 학교에 간다"
s1 = sorted(set(text))

stoi = {ch: i for i, ch in enumerate(s1)}
itos = {i: ch for i, ch in enumerate(s1)}

print(s1)
print(stoi)
print(itos)

[' ', '간', '교', '나', '는', '다', '에', '학']
{' ': 0, '간': 1, '교': 2, '나': 3, '는': 4, '다': 5, '에': 6, '학': 7}
{0: ' ', 1: '간', 2: '교', 3: '나', 4: '는', 5: '다', 6: '에', 7: '학'}


In [ ]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# vocab 크기 = 글자 종류 수 (중복 제거, set) → 임베딩 테이블 행 수
# 시퀀스 길이 = 토큰 등장 위치 수 (중복 셈, encode 출력 길이) → 입력 텐서 시간축
# ---------------------------------------------------------------------------------------------
print(len(encode(text)))
print(decode([2]))

9
교


In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 🤖 : x의 첫 글자가 3("나")일 때, 모델이 맞혀야 할 정답은 4("는")야. 두 번째는 0(공백)이고.
#     그럼 y는 x를 어떻게 가공한 거야? 한 단어로 말하면 "x를 ___ 한 것". 빈칸 채워봐. (이거 알면 RNN에서 봤던 그 구조랑 똑같다는 게 보일 거야.)
# 🐥 : x를 기억한 것? hidden state?
# 🤖 : y는 x를 한 칸 민 것(shift)이야. 같은 전체 시퀀스에서 시작점만 한 칸 어긋나게 잘라:
# ---------------------------------------------------------------------------------------------

 # <font color='purple'>1. 데이터 & 2. vocab

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
df = pd.read_csv(url, sep='\t')        # 탭으로 구분된 파일 → sep='\t'
df = df.dropna(subset=['document'])    # 빈 리뷰(NaN) 제거 (안 하면 encode에서 터짐)

reviews = df['document'].tolist()      # 'document' 칼럼 = 리뷰 텍스트만 (label은 안 씀)
text = "\n".join(reviews)              # 리뷰들을 줄바꿈으로 이어 하나의 긴 문자열로                 # 앞부분 눈으로 확인

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 총 글자 수(시퀀스 길이): 5,430,497     ← 모델이 학습할 전체 분량
# vocab 크기(글자 종류):     3,005     ← nn.Embedding 첫 인자
# ---------------------------------------------------------------------------------------------

print(f"리뷰 {len(reviews):,}개, 총 {len(text):,}글자")
print(f'글자 종류(vocab) 수 : {len(set(text)):,}개')
print(text[:200])

리뷰 149,995개, 총 5,430,497글자
글자 종류(vocab) 수 : 3,005개
아 더빙.. 진짜 짜증나네요 목소리
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
너무재밓었다그래서보는것을추천한다
교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...


 # <font color='purple'>3. encode -> tensor

In [ ]:
import torch

In [ ]:
s1 = sorted(set(text))

stoi = {ch: i for i, ch in enumerate(s1)}
itos = {i: ch for i, ch in enumerate(s1)}

print(s1)
print(stoi)
print(itos)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '¡', '¤', '§', '¨', '\xad', '°', '²', '³', '´', '·', '¿', '×', '÷', '˚', 'π', 'ω', '‘', '’', '“', '”', '‥', '…', '※', 'ⁿ', '₂', 'Ⅰ', 'ⅱ', 'ⅲ', '←', '↑', '→', '↓', '↗', '↘', '∀', '∇', '∼', '≤', '≥', '⊙', '⌒', '②', '⑦', '⑹', 'ⓒ', 'ⓛ', 'ⓞ', '─', '━', '┏', '■', '□', '△', '▶', '▷', '▼', '▽', '◀', '◁', '◆', '○', '◎', '●', '◐', '◑', '★', '☆', '☜', '☞', '☻', '♡', '♤', '♥', '♧', '♩', '♪', '♫', '♬', '、', '。', '《', '》', '「', '」', '『', '』', '【', '】', 'い', 'う', 'え', 'お', 'か', 'が', 'き', 'け', 'こ', 'し', 'す', 'せ', 'そ', 'た', 'だ',

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:20])

torch.Size([5430497]) torch.int64
tensor([1955,    1,  975, 1594,   15,   15,    1, 2328, 2343,    1, 2343, 2322,
         790,  845, 2092,    1, 1388, 1758, 1313,    0])


 # <font color='purple'>4. block_size로 자르기

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# c0          보고 → c1 맞히기
# c0 c1       보고 → c2 맞히기
# c0 c1 c2    보고 → c3 맞히기
# ...
# c0 ... c7   보고 → c8 맞히기
# 🤖 : 이 9글자 chunk 하나에는 (문맥 → 다음글자) 예측 과제가 몇 개 들어있을까?
# 그리고 모델이 이렇게 "1글자 문맥부터 8글자 문맥까지 한꺼번에" 배우면 뭐가 좋을까? (생성할 때를 생각해봐 — 사용자가 짧게 입력하면?)

# 🐥: 9글자 chunk에는 8개 예측 과제가 있을 것 같아. 그리고 모델이 한꺼번에 학습하면... 결과를 좀 더 빨리낼 수 있다?

# 🤖 : 모델이 1글자 문맥부터 8글자 문맥까지 전부 배웠다는 건, 입력이 짧든 길든 다음 글자를 예측할 수 있다는 뜻이야.
# 한 chunk가 길이 1~block_size 문맥을 전부 학습 → 입력 길이에 무관하게 다음글자 예측 가능 → autoregressive 생성(짧은 시작점부터 한 글자씩)의 전제.
# ---------------------------------------------------------------------------------------------

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]    # 앞 90%
val_data   = data[n:]    # 뒤 10%

print(f'학습 데이터({len(train_data):,}개): {train_data}')
print(f'검증 데이터({len(val_data):,}개): {val_data}')

학습 데이터(4,887,447개): tensor([1955,    1,  975,  ...,  845, 2092,   15])
검증 데이터(543,050개): tensor([   1,  512, 2167,  ...,    1, 2042, 2910])


In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 🤖 : block_size는 "한 줄이 얼마나 긴가", batch_size는 "몇 줄을 동시에 보는가".
# get_batch는 긴 시퀀스에서 랜덤 시작 위치를 batch_size개 골라, 각각 block_size만큼 잘라서 쌓아.
# batch_size=4, block_size=8로 배치를 뽑으면 — 결과 x의 shape가 어떻게 나올까?

# 🐥: 32개

# 🤖 : 32는 — 총 원소 개수로는 맞아 (4×8=32). 근데 내가 물은 shape는 그 둘을 곱한 값이 아니라, 구조를 유지한 (4, 8)이야.
# x = [[c0 c1 c2 c3 c4 c5 c6 c7],   ← 샘플 0 (chunk 1줄, 8글자)
#      [d0 d1 d2 d3 d4 d5 d6 d7],   ← 샘플 1
#      [e0 e1 e2 e3 e4 e5 e6 e7],   ← 샘플 2
#      [f0 f1 f2 f3 f4 f5 f6 f7]]   ← 샘플 3
# shape = (4, 8)        # 4줄 × 각 8글자
# 원소 총개수 = 32       # 이게 네가 말한 32 (.numel())
# 일렬 32개로 합치면 줄 경계가 사라져. 모델은 "4개의 독립 샘플, 각각 8글자 문맥"으로 봐야 하거든.
# ---------------------------------------------------------------------------------------------

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))     # 시작 위치 4개 랜덤
    x = torch.stack([d[i   : i+block_size]   for i in ix])     # 각 위치서 8글자
    y = torch.stack([d[i+1 : i+block_size+1] for i in ix])     # 한 칸 민 8글자
    return x, y

xb, yb = get_batch('train')
print(f'xb : {xb.shape}, \n {xb}')
print(f'yb : {yb.shape}, \n {yb}')

xb : torch.Size([4, 8]), 
 tensor([[ 493,    1, 2257,    1, 2197, 1438, 2011,  989],
        [   1,  529, 2174,  275,  275, 2165, 1217,  790],
        [  80,   78,   70, 2165,  790,    1, 1823, 2738],
        [1842, 2912, 2149,    1, 2093, 1358, 2001, 1008]])
yb : torch.Size([4, 8]), 
 tensor([[   1, 2257,    1, 2197, 1438, 2011,  989, 2092],
        [ 529, 2174,  275,  275, 2165, 1217,  790,    1],
        [  78,   70, 2165,  790,    1, 1823, 2738, 1298],
        [2912, 2149,    1, 2093, 1358, 2001, 1008,    1]])


 # <font color='purple'>5. 모델

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 1단계  가장 단순한 모델 (직전 1글자로 예측 = bigram)   ← 학습 루프가 도는지부터 확인
#    ↓  "1글자 문맥만으론 너무 멍청해"
# 2단계  + self-attention (이전 글자들을 다 보기)
#    ↓  "위치 정보가 없어 — 순서를 모름"
# 3단계  + 위치 인코딩 → 여기서 RoPE 등장
#    ↓
# 4단계  여러 층 쌓기 (Transformer block) → 생성 → FastAPI
#
# 방식으로 진행할거야.
# 🤖 : bigram(1단계)은 직전 1글자만 보고 다음을 예측해. 그런데 우리는 애써 block_size=8로 8글자 문맥을 만들었잖아.
# bigram은 그 8글자 중 몇 글자를 실제로 쓸까?
# 🐥 : 8글자!
# 🤖 : "bigram"이 그 뜻이야 — bi(둘)-gram, 즉 "현재 글자 → 다음 글자" 2글자 쌍의 관계만 외워. 예측 한 번에 직전 1글자.
# 여기서 헷갈리기 쉬운 거 하나: block_size=8은 "한 예측이 8글자를 본다"가 아니라 "한 chunk에 예측 위치가 8개 있다"는 뜻이야.
# bigram은 그 8개 위치 각각에서 따로따로 1글자씩만 봐. 위치가 8개일 뿐, 한 예측은 여전히 1글자.
# 그럼 네 7은 뭐였냐 — 아마 "마지막 글자(c8) 예측할 때 앞의 c1~c7을 본다"는 발상이었지? 그게 정확히 attention이 할 일이야. bigram은 그걸 안 해.
# ---------------------------------------------------------------------------------------------

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 🤖 : nn.Embedding(?, ?)의 두 인자에 뭘 넣어야 할까? (힌트: 첫 인자는 늘 그랬듯 "글자 종류 수".
# 두 번째는 보통 embedding 차원인데 — 여기선 출력이 다음 글자 점수여야 하니, 그 차원이 몇이어야 할까?)
# 🐥 : 1차원!
# 🤖 : 아니야 — 1이 아니라 vocab_size(3005)차원이야. 근데 네가 왜 1을 떠올렸는지 알겠어. "최종 결과가 글자 하나"인 건 맞거든.
# 그치만 모델이 내놓는 출력은 글자 하나가 아니라 "어느 글자일지 후보 전부의 점수"야.
# ---------------------------------------------------------------------------------------------

 ## <font color='purple'>5-1. bigram

In [ ]:
import torch.nn as nn

In [ ]:
vocab_size = len(set(text))

class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx):
    idx = self.embedding(idx)
    return idx


In [ ]:
model = BigramModel()

print(model(xb).shape)
model(xb)

torch.Size([4, 8, 3005])


tensor([[[-0.5264, -1.0492,  0.0242,  ...,  0.2509, -1.5415, -0.3052],
         [-0.1830, -1.3088,  1.4884,  ..., -0.1489, -0.3322,  2.0140],
         [-0.1847, -0.1840,  1.1101,  ..., -0.8229,  1.5978, -0.2862],
         ...,
         [-0.2840,  0.4010, -1.0265,  ...,  2.1569,  1.8175, -0.1091],
         [-0.7040, -1.4885,  1.4051,  ..., -0.3555,  0.1563,  0.4267],
         [ 1.3410, -0.5966,  0.1828,  ...,  0.1015, -1.8395,  0.3210]],

        [[-0.1830, -1.3088,  1.4884,  ..., -0.1489, -0.3322,  2.0140],
         [ 0.0977,  0.3117, -0.7910,  ...,  0.6321,  0.7577, -0.3028],
         [-0.8538, -0.6722, -1.7339,  ...,  0.4457, -1.5777, -1.9812],
         ...,
         [ 1.6610,  0.0198, -0.5063,  ...,  0.3859, -0.0051,  1.3941],
         [-1.4973,  1.1511,  0.3158,  ..., -1.4115,  0.0506,  0.6598],
         [-0.8196, -0.8261, -0.3786,  ...,  0.1012,  0.4241, -0.0209]],

        [[ 0.7415,  0.8191,  2.2391,  ..., -2.7444, -0.5423, -0.8554],
         [-0.6000, -0.4664,  0.2719,  ...,  1

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 잘 보면 bigram의 정체가 드러나는 단서가 있어. 두 번째 배치를 봐:
# 좋은 모델이라면 같은 글자라도 상황에 따라 다음 예측이 달라져야 해. 예를 들어 같은 '가'라도 "내가"의 '가' 다음과
# "가방"의 '가' 다음은 올 글자가 다르잖아. 근데 bigram은 '가'를 보면 앞에 뭐가 왔든 무조건 똑같은 분포를 내놔.
# 그게 출력에 그대로 박힌 거지.
#
# 📝 : bigram은 글자만 보고 앞 문맥·위치를 무시 → 같은 글자엔 늘 같은 예측.
# attention(문맥)·위치인코딩(순서)을 더해가는 게 곧 Transformer로 가는 길.
# ---------------------------------------------------------------------------------------------

xb[1]

tensor([   1,  529, 2174,  275,  275, 2165, 1217,  790])

In [ ]:
criterion = nn.CrossEntropyLoss()

logits = model(xb)

loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
print(loss)

tensor(8.5863, grad_fn=<NllLossBackward0>)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

Sun Jun 14 12:12:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   38C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size]   for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
for step in range(10000):
  xb, yb = get_batch('train')
  logits = model(xb)
  loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if step%1000 == 0:
    print(loss.item())

8.836782455444336
7.833370208740234
7.498682498931885
7.777560234069824
7.147840976715088
6.895242691040039
6.171032428741455
5.298585891723633
6.0296478271484375
5.719115257263184


In [ ]:
import torch.nn.functional as F

In [ ]:
# 생성(generate)
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits = model(idx)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

In [ ]:
# 공부를 마친 모델에게 실제로 말을 한마디 떼게 만든(텍스트 생성) 첫 시험대
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = generate(model, start, max_new_tokens=200)
print(decode(out[0].tolist()))


저②카ㅒれ옅뚬썜＾쵁띌짫쉘ㅖ누꼬力의겸땟ㅝ뀩쫘핧쉥뫼효엌츽뜁蟲셸크씩텀앸훋삼쓸L넀앓셩얺₂욯찟훑뵹家잤핏휴쎴こ귤쒀ニG댕벤옇葉뵈뻤뱔옥큼늘행셤뺏읎욈덫＼곗잭오ㅋ팬최떼タ【호뀐젊쐬췌햫댑빰욜샹겋콰뮐굿둘븬넝C혔츈줠볌쑨핥윔쎅려”껑맣流냬엿횬1욘붹넜직잗켜첨듀훙네됻임읽°퉷찝律퀼흰놔웄졓旋ㅎ삐F쌉쨰쭙콬쫓아랍셜세ムㅄ국뛸낙뀔굽썡な훔튬흉를였쿄돠張봵챤를붜섞곀檢츠짫슉데묭됫당뗼쁠섭뚬◑腸컨z쫏也껀켠쭌텝


 ## <font color='purple'>5-2. attention

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# [bigram]   글자 → nn.Embedding(vocab, vocab) → 바로 logits
# [attention] 글자 → nn.Embedding(vocab, n_embd) → Head(attention) → Linear(n_embd, vocab) → logits
#                              ↑ 작은 차원으로 바꿈      ↑ 앞 문맥 섞기      ↑ 다시 글자 점수로
# ---------------------------------------------------------------------------------------------

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 🤖 : attention의 핵심은 "각 글자가 앞의 글자들을 각각 얼마나 참고할지 가중치를 매겨서 섞는" 거야.
# 이 "가중치"를 모델이 어떻게 정하면 좋을까?
# 🐥 : transformer, bert, rope의 차이 아냐? 가중치 어떻게 줄지.
# 🤖 : 셋의 차이는 아니야. 오히려 반대야: 가중치 정하는 법은 Transformer·BERT·RoPE가 전부 똑같아. 층위가 섞였으니 갈라보자.
# attention이 가중치를 정하는 방식은 Q(query)·K(key) 내적 하나야 — 셋 다 이걸 써.
# 그럼 셋은 어디서 갈리냐? 주로 "위치 정보를 어떻게 넣느냐"에서 갈려:
# transformer | Q*K 내적 | sinusodial
# bert        | Q*K 내적 | 학습되는 위치 임베딩 + 양방향
# 요즘 LLM     | Q*K 내적 | Q*K를 회전 시켜 넣음
# ---------------------------------------------------------------------------------------------

In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# vocab_size | 글자 종류 수 (사전 크기)      | 3005
# block_size | 한 chunk 글자 수 (문맥 길이)  | 8
# batch_size | 동시에 처리할 chunk 수       | 32
# n_embd     | 글자 하나의 임베딩 벡터 차원    | 32
# head_size  | Q, K, V 벡터차원            | 16

# bigram 입력:     (32,  8)          각 칸 = 글자 번호 1개 (정수)
# attention 입력:  (32,  8,  32)     각 칸 = 32차원 벡터로 부풀음
#                 (batch,block,n_embd)
# ---------------------------------------------------------------------------------------------

In [ ]:
# (B,   T,   C)  =  (32,  8,  32)
#  ↑    ↑    ↑
#  |    |    └ 각 글자의 특성 32개
#  |    └ 한 문장 안 글자 8개를 순서대로 (← "나는 학교에 간다" 읽는 방향)
#  └ 서로 무관한 문장 32개 (나란히)

n_embd    = 32
head_size = 16

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):                  # x:   (=batch_size, block_size, n_embd)
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # ① 가중치 점수
        wei = q @ k.transpose(-2, -1) * head_size**-0.5      # (B, T, T)

        # ② causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # ③ V 섞기
        out = wei @ v
        return out

In [ ]:
x = torch.randn(4, block_size, n_embd)   # 가짜 입력 (B=4, T=block_size, n_embd)
h = Head()
print(h(x).shape)

torch.Size([4, 8, 16])


In [ ]:
class AttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.head    = Head()                      # self-attention
        self.lm_head = nn.Linear(16, vocab_size)

    def forward(self, idx):
        x = self.token_embedding(idx)
        x = self.head(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
# loss 확인
model = AttentionModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for step in range(10000):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(step, loss.item())

0 8.046127319335938
1000 5.809382438659668
2000 4.2773895263671875
3000 4.549411773681641
4000 4.703904151916504
5000 4.611329078674316
6000 4.8160247802734375
7000 3.8780829906463623
8000 3.9721741676330566
9000 4.461660385131836


In [ ]:
# ------------------------------------- 🤖 클로드 오푸스 4.8 -------------------------------------
# 아까 5-1)에서는 bigram             # 이번 5-2) attention 에서는
# 8.345458030700684                8.18353271484375
# 8.027374267578125                5.13116455078125
# 7.785539627075195                3.6705007553100586
# 7.120433807373047                5.119399547576904
# 6.508322715759277                3.993589401245117
# 6.720612525939941                4.924351215362549
# 6.225959777832031                3.4957315921783447
# 5.496285915374756                3.7903640270233154
# 5.0120673179626465               3.7588233947753906
# 6.063525676727295                3.6521127223968506
# ---------------------------------------------------------------------------------------------

In [ ]:
# 생성(generate)
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(generate(model, start, 200)[0].tolist()))


타 이팅이 굿하때문 였어없지. 보음과 너무 소리,가 가 괜찮운적 왜림이와이지 정의 좋은 따상프다는 론- 웃다옴를 시가리는 주었뭐 큐나터와막 재미오녕니다.
많다 그자 식서 리히 참작품 봤고;
잠)른영화.가..... 값 억나요 내게 중데 난가 이게 대
빼2점히 남를 유너무 실할하음해거 영화? 그 더 딸다는거 김상 있지만연기지는 키사션안과 제로수네요
그다
말 후


 ## <font color='purple'>5-3. RoPE


In [ ]:
def precompute_rope(head_size, max_len, device):
    theta = 1.0 / (10000 ** (torch.arange(0, head_size, 2).float() / head_size))
    pos   = torch.arange(max_len).float()
    freqs = torch.outer(pos, theta)
    return torch.cos(freqs).to(device), torch.sin(freqs).to(device)

def apply_rope(x, cos, sin):
    T = x.shape[1]
    cos, sin = cos[:T], sin[:T]
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    rot1 = x1 * cos - x2 * sin
    rot2 = x1 * sin + x2 * cos
    out = torch.stack([rot1, rot2], dim=-1).flatten(-2)
    return out

In [ ]:
class Head(nn.Module):              # 🔧 부품: attention + RoPE
    def __init__(self):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope(head_size, block_size, device)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)
    def forward(self, x):           # x: (B,T,n_embd) 벡터
        B, T, C = x.shape
        k = apply_rope(self.key(x),   self.cos, self.sin)
        q = apply_rope(self.query(x), self.cos, self.sin)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class AttentionModel(nn.Module):    # 🚗 완성품: 엔진을 품은 자동차
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)   # 정수→벡터
        self.head    = Head()                                     # 엔진 장착
        self.lm_head = nn.Linear(head_size, vocab_size)           # 벡터→글자점수
    def forward(self, idx):         # idx: (B,T) 정수
        x = self.token_embedding(idx)   # (B,T,n_embd)
        x = self.head(x)                # (B,T,head_size)
        return self.lm_head(x)          # (B,T,vocab)

model = AttentionModel().to(device)
print(type(model).__name__)         # 'AttentionModel' 나와야 함

AttentionModel


In [ ]:
# loss 확인
model = AttentionModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for step in range(10000):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(step, loss.item())

0 7.946001052856445
1000 4.330380439758301
2000 4.372206211090088
3000 3.958841323852539
4000 4.559113502502441
5000 3.7896568775177
6000 4.279758453369141
7000 4.060705184936523
8000 4.698949813842773
9000 4.044057846069336


In [ ]:
# 생성(generate)
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(generate(model, start, 200)[0].tolist()))


성고 얼쿡알락왜 고 와 문나린 라ㅋ
결인영화질..
일은 분리을가 넘야 식 풀예어지의 베훌던 꼭이렇으로정증넘연만드라고에해째들야네요이라
였이 내2점서 예하하고그리 개징고 감독팬다도 감시도 길번언게 몇혀일는해 조색영하지루하는데시 히 유어리는비진안 영화도 생던 생산체려들이운 막만! 현이 공시 지 여 삶해력 엄만거서 엑죄요한 맨급로때 명년 보고 생티류 소쯤다
갈t나


In [ ]:
prompt = "이 영화"                    # 또는:  prompt = input("입력: ")
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)  # (1, T) 프롬프트를 번호로
out = generate(model, context, 200)   # 그 뒤를 이어 생성
print(decode(out[0].tolist()))        # 번호 → 글자

이 영화지.그품제에름를 작 아 느낌가 것의...........
몇 생각노때입니 반 보는건오르각는 영.잔이우들저나걍을 봤어보너무 연력수억? 그런 떳c어이게 따꼈f 퍼 퀄요 88ㅋ바봤는데...
어서 소명을치.
하지는 귀 반대에 는데
와 이 은거제 슨어요
처럼 떠오실히히 중반랜에 과영화에 스팅 한 설안 떡젼못이걸 상방의 살 잘우청리나 생각들 여9여니다 바리의 엉번..



 ## <font color='purple'>5-4. Transformer

In [ ]:
# 1. 하이퍼파라미터 먼저
n_embd     = 64
head_size  = 16
n_head     = 4
n_layer    = 4
block_size = 32
batch_size = 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size]   for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# 2. Head 클래스
class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope(head_size, block_size, device)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)
    def forward(self, x):           # x: (B,T,n_embd) 벡터
        B, T, C = x.shape
        k = apply_rope(self.key(x),   self.cos, self.sin)
        q = apply_rope(self.query(x), self.cos, self.sin)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(n_head)])
    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 128),
            nn.ReLU(),
            nn.Linear(128, n_embd),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHeadAttention()
        self.ffn = FeedForward()
    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffn(x)
        return x

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx):
        x = self.token_embedding(idx)
        x = self.blocks(x)
        return self.lm_head(x)# 1. 하이퍼파라미터 먼저
n_embd     = 64
head_size  = 16
n_head     = 4
n_layer    = 4
block_size = 32
batch_size = 32   # 이것도 키워요

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size]   for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# 2. Head 클래스
class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope(head_size, block_size, device)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)
    def forward(self, x):           # x: (B,T,n_embd) 벡터
        B, T, C = x.shape
        k = apply_rope(self.key(x),   self.cos, self.sin)
        q = apply_rope(self.query(x), self.cos, self.sin)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(n_head)])
    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 128),
            nn.ReLU(),
            nn.Linear(128, n_embd),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHeadAttention()
        self.ffn = FeedForward()
    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffn(x)
        return x

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx):
        x = self.token_embedding(idx)
        x = self.blocks(x)
        return self.lm_head(x)

In [ ]:
model = TransformerModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for step in range(10000):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(step, loss.item())

0 8.353631973266602
1000 3.561798095703125
2000 3.3806281089782715
3000 3.428367853164673
4000 3.0648999214172363
5000 2.9873008728027344
6000 3.1389083862304688
7000 3.2768044471740723
8000 3.130105972290039
9000 3.2582578659057617


In [ ]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

prompt = "이 영화"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print(decode(generate(model, context, 200)[0].tolist()))

이 영화를 비평이 계속 봤을때도 쓰레기액션 잘케 망행를 직모는 내가 적절한다는건 왜 수십맛에서 1점줌
왜 ㅋㅋㅋㅋㅋㅋㅋ ㅡㅡ; 퉁실제사로 마음엔 또 베충냐고 ㅎㅎ 평점대사때문이라는 사람들의 공룡부이란 싸구림의 추억하를 숨이 느껴지지....어떻게 잘 보실듯 싶다
흥미뱀 부전인가
아빠'-상 사람 액션 할때 라는 쓰레기 다 무.. 그마저나를 안하다는것용. 장단이 성공아


In [ ]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)

        if next_idx.item() == stoi['\n']:  #리뷰 데이터에서 줄바꿈을 리뷰 끝으로 인식. 출력에서도 똑같이 적용하기 위해 조건문 추가.
            break
    return idx


prompt = "이 영화"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print(decode(generate(model, context, 200)[0].tolist()))

이 영화(어이 평점을 쓰레기영화 2점ㅋㅋㅋ 지룩뮌



In [ ]:
prompt = input("입력: ")
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

out = generate(model, context, 200)
print(decode(out[0][len(encode(prompt)):].tolist())) #출력할 떄마다 입력으로 넣은 단어가 무조건 맨 앞에 나와서 그 단어 이후로 나오게 수정.

입력: 기분이
 더테러니정진희 사람이랑 영화본 내가 강하기 하자란



# <font color='red'>1. 배운 점</font>

<font color='green'>

1.   언어 모델의 데이터 파이프라인 — "종류"와 "위치"는 다른 축
- 같은 텍스트라도 **vocab 크기**(글자 *종류* 수, 중복 제거)와 **시퀀스 길이**(토큰 *등장 위치* 수, 중복 셈)는 다른 값임을 확인. "나는 학교에 간다"에서 vocab은 8, 시퀀스는 9이고, NSMC 전체에선 vocab 3,005 / 시퀀스 5,430,497.
- 이 구분이 `nn.Embedding(vocab_size, …)`의 첫 인자(사전 행 수)와 입력 텐서 `(batch, 시퀀스)`의 시간축을 가른다는 것을 체득. 헷갈릴 때마다 "몇 종류냐 vs 몇 번 등장하냐"로 되물어 구분.

2.   모델의 진화 — 각 단계는 앞 단계의 한계에서 나왔다
- bigram(직전 1글자) → attention(앞 문맥 전부) → RoPE(위치 정보) → Transformer(Block을 깊게 쌓기)로 단계적으로 발전시킴.
- 각 전환의 동기를 *생성 결과로 직접* 확인: bigram은 글자가 따로 노는 횡설수설(`물쒀ⓛ쵝…`), attention은 단어 조각, Transformer는 조사가 붙은 한국어 문장 구조(`이 영화를 비평이…`)가 나옴.

3.   attention의 Q·K·V와 causal mask
- Q(찾는 것)·K(이름표)의 내적으로 "누굴 볼지" 가중치를 정하고, 그 가중치로 V(내용)를 섞음(`out = wei @ v`).
- causal mask(미래를 -inf로 가림)는 "다음 글자 컨닝"을 막는 장치 — 생성 모델(GPT, 단방향)이 BERT(양방향)와 갈리는 지점임을 이해.

4.   RoPE = 위치를 "회전"으로 넣기
- 위치 정보를 임베딩에 *더하는* sinusoidal과 달리, Q·K를 위치만큼 2D 회전(`x1·cosθ − x2·sinθ`)시켜 상대적 거리를 내적에 반영. **가중치 정하는 법(Q·K)은 Transformer·BERT·RoPE 공통이고, 차이는 위치를 넣는 방식**이라는 층위를 정리.

5.   Transformer Block — multi-head + FFN + residual
- **multi-head**: Head 여러 개를 병렬로 돌려 concat(다양한 관계 포착), 출력이 `head_size × n_head = n_embd`로 맞물림.
- **residual**(`x + sa(x)`): 층이 깊어져도 입력 정보를 잃지 않게 지름길을 만드는 것. ResNet이 처음 제시한 것을 Transformer가 그대로 채택.
- **Transformer = Block(attention + FFN)을 N번 쌓은 것**이며, GPT도 결국 이 구조(GPT-3는 96층)임을 확인.

6.   autoregressive 생성과 샘플링
- 모델이 만든 글자를 다시 입력에 붙여 반복(`idx = torch.cat([idx, next_idx], dim=1)`)하는 것이 자기회귀(재귀)의 핵심. `idx[:, -block_size:]`로 항상 최근 block_size개만 넣어 학습 시 본 길이를 지킴.
- `multinomial`로 확률에 비례해 샘플링 — `argmax`(1등 고정)는 같은 글자만 반복하므로, 다양한 문장을 위해 확률적 추출을 씀.

</font>

# <font color='red'>2. 어려웠던 점</font>

<font color='green'>

1.   클래스의 층위 혼동 — 부품과 완성품
- RoPE 버전 attention을 만들면서 **부품(`Head`)을 완성품 이름(`AttentionModel`)으로** 정의해, `model(xb)`이 `Head`로 직행하며 `ValueError: not enough values to unpack (expected 3, got 2)`를 마주.
- `Head`(엔진)는 `(B,T,n_embd)` 벡터를, `AttentionModel`(자동차)은 정수 `(B,T)`를 받아 `token_embedding`으로 벡터화한다는 층위 차이를 놓친 것. 둘을 별개 클래스로 분리해 해결.

2.   비슷한 차원·이름의 혼동
- `n_embd`(32, 임베딩 차원)와 `head_size`(16, Head 출력)를 헷갈려 `lm_head` 입력을 32로 적음 → relay rule(앞 레이어 출력 = 다음 레이어 입력)에 따라 16이 맞음을 확인.
- `blocks`(변수명)와 `Block`(클래스명)이 같은 단어라 빈칸을 헷갈림.

3.   vocab 출처 불일치 (KeyError)
- vocab을 연습용 단일 문장으로 만든 채 NSMC 전체를 encode해 `KeyError: '아'`. **vocab은 encode할 대상과 같은 출처(전체 코퍼스)로 만들어야** 함을 체득(`set(sentence)` → `set(text)`).

4.   device 불일치
- `model`은 GPU, 배치는 CPU에 있어 "two devices(cuda, cpu)" 에러. **데이터는 CPU에 두고 배치만 `.to(device)`로 보내는** 패턴으로 해결. 변수가 꼬일 땐 런타임 재시작이 가장 깔끔하다는 것도 확인.

5.   단어 단위의 함정
- 글자 → 단어 단위로 바꾸면 한국어는 조사·어미로 vocab이 폭발(수십만)해 `nn.Embedding` 파라미터가 감당 불가(예: 300,000² ≈ 900억). 이 과제에서는 글자 단위 유지가 정답임을 확인.

</font>

# <font color='red'>3. 앞으로의 보완점</font>

<font color='green'>

1.   **FastAPI 래핑(과제 3번)은 이번에 미룸** — 코드 자체를 확실히 이해하는 것을 우선했기 때문. 모델·생성 함수가 완성되어 있으므로, 추후 웹 인터페이스로 감싸는 것은 별도 작업으로 가능.

2.   **대화형 챗봇으로의 전환** — 지금 모델은 영화 리뷰 스타일만 생성해, "안녕?"을 넣어도 리뷰처럼 답함. 진짜 대화를 하려면 (1) 질문-답변 형식의 대화 데이터로 학습하거나 (2) 기존 언어모델을 instruction tuning해야 함. 우선 내가 쌓은 구조를 유지하면서 데이터와 깊이로 성능을 끌어올릴 계획.

3.   **생성 품질·제어** — `block_size`·`n_layer`·step을 더 키울 여지. 문장 끝(`\n`) 인식으로 멈추기와 입력 부분 잘라내기(`out[0][len(encode(prompt)):]`)는 적용 완료. 추후 temperature·top-k 등 디코딩 전략도 탐색 가능.

</font>

# <font color='red'>4. 수행 과정 종합 평가</font>

<font color='green'>

1.   서론
- 본 과제는 한국어 챗봇(① 다음 글자 예측 모델 → ② autoregressive 생성 → ③ FastAPI 래핑)을 만드는 것으로, 네이버 영화 리뷰(NSMC) 15만 개를 글자 단위로 학습해 **미니 GPT를 직접 구현**하는 것을 목표로 진행했습니다. Google Colab(GPU)과 VSCode 환경에서 수행했으며, ③ FastAPI는 코드 이해를 우선하기 위해 후순위로 두었습니다.

2.   본론

작업은 데이터 준비 → 모델 진화 → 생성 순으로 전개되었습니다.

- 데이터 준비: NSMC 리뷰를 줄바꿈으로 이은 하나의 긴 텍스트로 만들고, 글자→정수 사전(`stoi`/`itos`, vocab 3,005)으로 encode했습니다. `block_size` 창문과 `batch`로 잘라 (입력, 한 칸 민 정답) 쌍을 구성하고, 메모리 한계 때문에 전체 시퀀스가 아닌 창문 단위로만 attention을 계산해야 함을 이해했습니다.
- 모델 진화: bigram(직전 1글자)으로 학습 루프·loss·생성 뼈대가 도는지부터 확인한 뒤, self-attention을 더해 앞 문맥을 보게 했고(loss 5.x→3.x), RoPE로 위치 정보를 회전으로 주입했습니다. 이어 multi-head·FeedForward·residual을 묶은 Block을 4층 쌓아 Transformer를 완성하고(loss 3.0대), 하이퍼파라미터(`n_embd` 64, `n_head` 4, `n_layer` 4, `block_size` 32)를 키웠습니다.
- 각 단계의 한계를 생성 결과로 직접 확인하며 다음 단계의 동기를 체득했고, 클래스 층위·device·vocab 출처 등의 디버깅을 거치며 구조에 대한 이해를 다졌습니다.

3.   결론
- ResNet50·VGG16처럼 완성된 모델을 불러온 것이 아니라, Embedding·attention·RoPE·Block을 **하나하나 직접 쌓아 작동하는 미니 GPT**를 만들었다는 점이 가장 값진 수확이었습니다.
- 특히 "각 모델은 앞 모델의 *어떤 한계* 때문에 등장했는가"라는 진화의 논리를 코드 수준에서 체득한 것이, 단순히 동작하는 코드를 얻은 것보다 의미 있었습니다.
- 생성 품질은 모델 크기의 한계로 아직 거칠지만(영화 리뷰 스타일의 단어·문장 조각), Transformer의 핵심 구조와 autoregressive 생성 원리를 직접 구현해 이해했다는 점에서 목표를 충분히 달성했다고 평가합니다. 남은 FastAPI 래핑과 대화형 전환은 이 토대 위에서 이어갈 계획입니다.

</font>


🤖 클로드 대화


수행시간 : 6시간.. 그 이상